# Crawl TopCV từ sitemap

Hai bước: lấy link IT/CNTT/Data Science năm 2026 từ sitemap và lưu TXT, sau đó crawl chi tiết từ TXT. Mỗi tin được ghi ngay vào JSONL với đúng 14 cột gốc của TinixAI. Chạy lại bước crawl để tiếp tục các tin chưa tải thành công.


## 1. Nạp crawler và xác định thư mục dự án

In [ ]:
import importlib
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / 'src' / 'crawlers' / 'topcv_it_jobs.py').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Hãy mở notebook từ thư mục dự án hoặc thư mục notebooks/.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.crawlers import topcv_it_jobs as crawler
crawler = importlib.reload(crawler)
print('Dự án:', PROJECT_ROOT)
print('JSONL:', crawler.RAW_JSONL)

## 2. Cấu hình

Chỉ cần chọn năm, số luồng và thời gian chờ. Đường dẫn lưu do crawler quản lý.
Chạy cell lấy link khi muốn quét mới; khi đã có file link thì bỏ qua cell đó và chạy cell crawl.


In [ ]:
YEAR = 2026  # Năm lastmod, không phải năm đăng tin
WORKERS = 2
DELAY = 2.0

print('File link:', crawler.job_urls_file(YEAR))
print('Dữ liệu thô:', crawler.RAW_JSONL)


## 3. Lấy link từ sitemap và lưu TXT

Chỉ giữ URL có từ khóa IT/CNTT/Data Science và lastmod thuộc YEAR. Chạy lại cell này để cập nhật file link.
Nếu đã có file link đã lọc thì bỏ qua cell này. Bước crawl bên dưới chỉ đọc file.

Slug có thể không nêu đầy đủ nghề nghiệp; bộ lọc chỉ giữ URL có từ khóa nhận diện được.


In [ ]:
job_urls = crawler.get_job_urls_from_sitemap(YEAR, delay=DELAY)
display(pd.DataFrame({'url': job_urls[:10]}))


## 4. Crawl chi tiết từ file link

Ô này chỉ đọc link đã lưu, không quét lại sitemap hay trang danh sách. Có thể chạy trong một phiên kernel mới sau các ô thiết lập/cấu hình. Khi dừng giữa chừng, các tin đã ghi vẫn nằm trong JSONL để lần sau tiếp tục. Mỗi bản ghi dùng schema TinixAI; `year` lấy từ `datePosted` trên trang tin, không lấy từ sitemap.


In [ ]:
job_urls = crawler.load_job_urls(YEAR)
crawl_complete = crawler.crawl_from_links(job_urls, workers=WORKERS, delay=DELAY)
print('Lượt crawl hoàn tất:', crawl_complete)


## 5. Chuyển JSONL sang CSV và Parquet

Chạy sau cell nạp crawler; không cần chạy lại sitemap hoặc crawl. Xuất đúng 14 cột gốc của TinixAI theo cùng thứ tự. Dòng JSON lỗi do crawl bị ngắt sẽ được bỏ qua và báo số lượng.
Kết quả lưu ở `data/processed/topcv_jobs.csv` và `data/processed/topcv_jobs.parquet`; chạy lại cell sẽ ghi đè hai file này.


In [ ]:
import json
import pyarrow as pa

records = []
invalid_lines = 0
with crawler.RAW_JSONL.open(encoding="utf-8-sig") as file:
    for line in file:
        if not line.strip():
            continue
        try:
            record = json.loads(line)
        except json.JSONDecodeError:
            invalid_lines += 1
            continue
        if not isinstance(record, dict) or set(record) != set(crawler.TINIXAI_COLUMNS):
            raise ValueError("JSONL phải chứa đúng 14 cột TinixAI; hãy kiểm tra file dữ liệu.")
        records.append(record)

print(f"Đọc {len(records):,} bản ghi; bỏ qua {invalid_lines:,} dòng lỗi.")
if not records:
    raise ValueError("Không có bản ghi hợp lệ để xuất.")

jobs = pd.DataFrame.from_records(records, columns=crawler.TINIXAI_COLUMNS)
output_dir = PROJECT_ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
csv_file = output_dir / "topcv_jobs.csv"
parquet_file = output_dir / "topcv_jobs.parquet"

jobs.to_csv(csv_file, index=False, encoding="utf-8-sig")
jobs.to_parquet(
    parquet_file, index=False,
    schema=pa.schema([
        (column, pa.int64() if column in ("id", "year") else pa.string())
        for column in crawler.TINIXAI_COLUMNS
    ]),
)

print("CSV:", csv_file)
print("Parquet:", parquet_file)
display(jobs.head())
